# TwinAIR data demo

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import scdata as sc
from scdata.models import Metric
import pandas as pd

## Device

### Load data

In [ ]:
device = sc.Device(blueprint='sc_air', params=sc.APIParams(id=18061))

# Load
device.backup_load()

In [ ]:
# Device info
device.handler.json # Information (metada) on api.smartcitizen.me -> api.smartcitizen.me/v0/devices/18601

print (device.handler.json.last_reading_at) # Last date of recorded data in SC API (in UTC)
print (device.data.index[-1]) # Last date of back-up stored (updated every 20 days)

### Data channels

In [ ]:
# Columns
print (device.data.columns)
print ('---')
for sensor in device.sensors:
    print (sensor.name, sensor.unit)

### Data processing

In [ ]:
device.data
type(device.data)

In [ ]:
# Basic correction
device.data['SCD30_CO2_MULT'] = device.data['SCD30_CO2']*30

In [ ]:
device.data.loc[:, ['SCD30_CO2_COR', 'SCD30_CO2']]

#### Metrics

In [ ]:
device.metrics

In [ ]:
for metric in device.metrics:
    print (metric.name, metric.function)

In [ ]:
device.data.columns

In [ ]:
help(sc.device.process.timeseries.poly_ts)

In [ ]:
# A*TEMP^C + B*SCD30_TEMP^D
metric = Metric(name='Temp difference',
                description='Basic Polynomial calculation',
                function='poly_ts',
                kwargs= {'channels': ['TEMP', 'SCD30_TEMP'],
                         'coefficients': [1, -1]}
               )

device.add_metric(metric)

In [ ]:
device.process()

In [ ]:
device.data.loc[:, ["Temp difference", "TEMP", "SCD30_TEMP"]]

#### Baseline

In [ ]:
help(sc.device.process.timeseries.baseline_als)

In [ ]:
# A*TEMP^C + B*SCD30_TEMP^D
metric = Metric(name='SCD30_CO2_baseline',
                description='Baseline for CO2',
                function='baseline_als',
                kwargs= {'name': 'SCD30_CO2'})
device.add_metric(metric)

In [ ]:
device.process()

In [ ]:
device.data.loc[:, ['SCD30_CO2', 'SCD30_CO2_baseline']]

#### NO2 + O3 Calculations

In [ ]:
device_2 = sc.Device(params=sc.APIParams(id=17558))

In [ ]:
device_2.handler.postprocessing

In [ ]:
device_2.handler.hardware_postprocessing

In [ ]:
for metric in device_2.metrics:
    print (metric.name, metric.kwargs, metric.function)

In [ ]:
device_2.backup_load()

In [ ]:
device_2.loaded = True

In [ ]:
for col in device_2.data.columns:
    print (col)

In [ ]:
device_2.process()

In [ ]:
device_2.data.loc[:, 'NO2']

### Backup

In [ ]:
# device.backup(format='parquet', mode='append', path='processed')

## Test (group of devices)

In [ ]:
devices = pd.read_excel("T6.1 - TwinAIR Pilots - Sensor Deployments.xlsx", sheet_name="DEVICES")

## Devices locations

In [ ]:
ciit_devices = devices[devices.Location == 'CIIT']

In [ ]:
device_ids=list(ciit_devices.id.values[:-1])
device_ids

### Create test

In [ ]:
test = sc.Test(name='CIIT_TWINAIR',
            devices=[sc.Device(blueprint='sc_air',
                               params=sc.APIParams(id=id)) for id in device_ids]
             )

In [ ]:
test.devices

In [ ]:
for device in test.devices:
    device.backup_load()

### Metric baseline

In [ ]:
metric = Metric(
    name='SCD30_CO2_baseline',
    description='Calculation of CO2 correction against its baseline',
    function='baseline_als',
    unit='ppm',
    post=True,
    args=None,
    kwargs={'name': 'SCD30_CO2', 'lam': 10000000}
)

for device in test.devices:
    device.add_metric(metric)

test.process()

baseline_co2_conc = 450

for device in test.devices:
    device.data['SCD30_CO2_COR'] = device.data['SCD30_CO2']- device.data['SCD30_CO2_baseline'] + baseline_co2_conc

In [ ]:
test.get_device(18061).data

### Plot

In [ ]:
traces = {
            "1": {"devices": [18061],
                  "channel": "SCD30_TEMP",
                  "subplot": 1},
            "2": {"devices": [18061, 18062],
                  "channel": "TEMP",
                  "subplot": 1}
        }

test.ts_uplot(traces = traces,
              formatting = {'width':1000,
                            'title': 'CO2 Comparison',
                            'ylabel': {1: 'CO2 [ppm]'}
                           },
              options = {'frequency': '30Min',
                         'min_date': '2024-07-31'})

In [ ]:
traces = {
            "1": {"devices": "all",
                  "channel": ["SCD30_CO2", "SCD30_CO2_baseline"],
                  "subplot": 1}
        }

test.ts_uplot(traces = traces,
              formatting = {'width':1000,
                            'title': 'CO2 Baseline',
                            'ylabel': {1: 'CO2 [ppm]'}
                           },
              options = {'frequency': '30Min',
                         'min_date': '2024-07-31'})